# FSOC Beacon Detection - YOLOv11 Training on Google Colab

This notebook trains a YOLOv11 model to detect beacons in the FSOC PAT simulation.

**Dataset:** 10,000 images (6,670 train / 1,670 val / 1,660 test)

**Split:** 67% train / 17% val / 17% test

## Step 1: Setup Environment

In [ ]:
# Check GPU availability
!nvidia-smi

In [ ]:
# Install Ultralytics YOLOv11
!pip install ultralytics

In [ ]:
# Import libraries
import os
from pathlib import Path
from ultralytics import YOLO
import yaml
import shutil

print("✓ Environment ready")

## Step 2: Upload Dataset

**Option A: Upload ZIP file**
1. ZIP the `dataset/` folder on your local machine
2. Upload `dataset.zip` using the file browser (left sidebar)
3. Run the cell below to extract

**Option B: Mount Google Drive**
1. Upload `dataset.zip` to your Google Drive
2. Mount Drive and copy the dataset

In [ ]:
# Option A: Extract uploaded ZIP
!unzip -q dataset.zip -d .
print("✓ Dataset extracted")

In [ ]:
# Option B: Mount Google Drive (alternative)
# from google.colab import drive
# drive.mount('/content/drive')
# !cp /content/drive/MyDrive/dataset.zip .
# !unzip -q dataset.zip -d .

## Step 3: Verify Dataset

In [ ]:
# Verify dataset structure
dataset_path = Path('dataset')

# Count images
train_images = len(list((dataset_path / 'images' / 'train').glob('*.png')))
val_images = len(list((dataset_path / 'images' / 'val').glob('*.png')))
test_images = len(list((dataset_path / 'images' / 'test').glob('*.png')))

# Count labels
train_labels = len(list((dataset_path / 'labels' / 'train').glob('*.txt')))
val_labels = len(list((dataset_path / 'labels' / 'val').glob('*.txt')))
test_labels = len(list((dataset_path / 'labels' / 'test').glob('*.txt')))

print("Dataset Verification:")
print("="*50)
print(f"Train: {train_images} images, {train_labels} labels")
print(f"Val:   {val_images} images, {val_labels} labels")
print(f"Test:  {test_images} images, {test_labels} labels")
print(f"Total: {train_images + val_images + test_images} images")
print("="*50)

# Verify counts match
assert train_images == train_labels, "Train image-label mismatch!"
assert val_images == val_labels, "Val image-label mismatch!"
assert test_images == test_labels, "Test image-label mismatch!"
print("✓ All image-label counts match")

# Check dataset.yaml exists
assert (dataset_path / 'dataset.yaml').exists(), "dataset.yaml not found!"
print("✓ dataset.yaml found")

In [ ]:
# Display dataset.yaml
with open('dataset/dataset.yaml', 'r') as f:
    print(f.read())

In [ ]:
# Display a sample label
sample_label = list((dataset_path / 'labels' / 'train').glob('*.txt'))[0]
print(f"Sample label: {sample_label.name}")
print(f"Content: {sample_label.read_text()}")
print("\nFormat: <class> <x_center> <y_center> <width> <height>")
print("All coordinates normalized to [0, 1]")

## Step 4: Visualize Sample Images

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np

def visualize_sample(image_path, label_path):
    """Visualize image with ground truth bounding box."""
    # Read image
    img = cv2.imread(str(image_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]

    # Read label
    with open(label_path, 'r') as f:
        label = f.read().strip().split()

    # Parse YOLO format
    class_id, x_center, y_center, width, height = map(float, label)

    # Convert to pixel coordinates
    x1 = int((x_center - width/2) * w)
    y1 = int((y_center - height/2) * h)
    x2 = int((x_center + width/2) * w)
    y2 = int((y_center + height/2) * h)

    # Draw bounding box
    cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
    cv2.circle(img, (int(x_center * w), int(y_center * h)), 3, (255, 0, 0), -1)

    return img

# Visualize 6 random samples
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.ravel()

train_imgs = list((dataset_path / 'images' / 'train').glob('*.png'))[:6]
for idx, img_path in enumerate(train_imgs):
    label_path = dataset_path / 'labels' / 'train' / f"{img_path.stem}.txt"
    img = visualize_sample(img_path, label_path)
    axes[idx].imshow(img)
    axes[idx].set_title(img_path.name)
    axes[idx].axis('off')

plt.tight_layout()
plt.show()
print("✓ Green box = Ground truth, Blue dot = Center")

## Step 5: Train YOLOv11 Model

Training configurations:
- **Model:** YOLOv11n (nano - fastest, smallest)
- **Epochs:** 100 (can increase for better accuracy)
- **Image Size:** 640×640
- **Batch Size:** 16 (adjust based on GPU memory)
- **Optimizer:** AdamW
- **Data Augmentation:** Auto (YOLOv11 default)

In [ ]:
# Load pretrained YOLOv11n model
model = YOLO('yolo11n.pt')
print("✓ Loaded YOLOv11n pretrained weights")

In [ ]:
# Train YOLO11n for 30 epochs
results = model.train(
    data='/content/fsoc_dataset/dataset.yaml',
    epochs=30,
    imgsz=640,
    batch=16,
    name='beacon_detector_30ep',
    patience=8,
    save=True,
    device=0,
    workers=2,
    project='/content/runs/detect',
    exist_ok=False,
    optimizer='AdamW',
    verbose=True,
    seed=42,
    deterministic=False,
    single_cls=True,
    rect=False,
    cos_lr=True,
    close_mosaic=5,
    amp=True,
    fraction=1.0,
    val=True,
    plots=True
)

print("\n✓ Training complete!")

## Step 6: Evaluate Model

In [ ]:
# Validate on test set
metrics = model.val(
    data='dataset/dataset.yaml',
    split='test',
    imgsz=640,
    batch=16,
    save_json=True,
    save_hybrid=True,
    conf=0.001,
    iou=0.6,
    max_det=100,
    plots=True
)

print("\nTest Set Metrics:")
print("="*50)
print(f"mAP50: {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.p[0]:.4f}")
print(f"Recall: {metrics.box.r[0]:.4f}")
print("="*50)

## Step 7: Visualize Training Results

In [ ]:
# Display training curves
from IPython.display import Image, display

results_dir = Path('runs/detect/beacon_detector')

print("Training Results:")
display(Image(filename=str(results_dir / 'results.png')))

print("\nConfusion Matrix:")
display(Image(filename=str(results_dir / 'confusion_matrix.png')))

print("\nPrediction Examples:")
display(Image(filename=str(results_dir / 'val_batch0_pred.jpg')))

## Step 8: Test Inference

In [ ]:
# Load best trained model
best_model = YOLO('runs/detect/beacon_detector/weights/best.pt')
print("✓ Loaded best model")

In [ ]:
# Run inference on test images
test_imgs = list((dataset_path / 'images' / 'test').glob('*.png'))[:6]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.ravel()

for idx, img_path in enumerate(test_imgs):
    # Run inference
    results = best_model(img_path, conf=0.25, verbose=False)

    # Plot result
    result_img = results[0].plot()
    result_img = cv2.cvtColor(result_img, cv2.COLOR_BGR2RGB)

    axes[idx].imshow(result_img)
    axes[idx].set_title(f"{img_path.name}\nConf: {results[0].boxes.conf[0]:.2f}" if len(results[0].boxes) > 0 else f"{img_path.name}\nNo detection")
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

## Step 9: Export Model

In [ ]:
# Export to ONNX format (for deployment)
best_model.export(
    format='onnx',
    imgsz=640,
    dynamic=False,
    simplify=True,
    opset=12
)

print("✓ Model exported to ONNX")
print("  Location: runs/detect/beacon_detector/weights/best.onnx")

In [ ]:
# Download trained model
from google.colab import files

# Download PyTorch model
files.download('runs/detect/beacon_detector/weights/best.pt')

# Download ONNX model
files.download('runs/detect/beacon_detector/weights/best.onnx')

print("✓ Models downloaded")

## Step 10: Performance Analysis

In [ ]:
# Analyze model performance
import pandas as pd

# Read training results
results_csv = pd.read_csv('runs/detect/beacon_detector/results.csv')
results_csv = results_csv.rename(columns=lambda x: x.strip())

print("Training Summary:")
print("="*50)
print(f"Total Epochs: {len(results_csv)}")
print(f"Best mAP50: {results_csv['metrics/mAP50(B)'].max():.4f}")
print(f"Best mAP50-95: {results_csv['metrics/mAP50-95(B)'].max():.4f}")
print(f"Final Train Loss: {results_csv['train/box_loss'].iloc[-1]:.4f}")
print(f"Final Val Loss: {results_csv['val/box_loss'].iloc[-1]:.4f}")
print("="*50)

# Plot training curves
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Loss curves
axes[0, 0].plot(results_csv['train/box_loss'], label='Train Box Loss')
axes[0, 0].plot(results_csv['val/box_loss'], label='Val Box Loss')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].set_title('Box Loss')
axes[0, 0].legend()
axes[0, 0].grid(True)

# mAP curves
axes[0, 1].plot(results_csv['metrics/mAP50(B)'], label='mAP50', color='green')
axes[0, 1].plot(results_csv['metrics/mAP50-95(B)'], label='mAP50-95', color='blue')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('mAP')
axes[0, 1].set_title('Mean Average Precision')
axes[0, 1].legend()
axes[0, 1].grid(True)

# Precision/Recall
axes[1, 0].plot(results_csv['metrics/precision(B)'], label='Precision', color='orange')
axes[1, 0].plot(results_csv['metrics/recall(B)'], label='Recall', color='purple')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Score')
axes[1, 0].set_title('Precision & Recall')
axes[1, 0].legend()
axes[1, 0].grid(True)

# Learning rate
axes[1, 1].plot(results_csv['lr/pg0'], label='Learning Rate', color='red')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('LR')
axes[1, 1].set_title('Learning Rate Schedule')
axes[1, 1].legend()
axes[1, 1].grid(True)

plt.tight_layout()
plt.show()

## Summary

✅ **Training Complete!**

- Dataset: 10,000 images (6,670 train / 1,670 val / 1,660 test)
- Model: YOLOv11n (nano)
- Training: 100 epochs with early stopping
- Metrics: mAP, Precision, Recall calculated on test set
- Export: PyTorch (.pt) and ONNX (.onnx) formats

**Next Steps:**
1. Download trained model (best.pt or best.onnx)
2. Integrate into FSOC simulation backend
3. Replace simple centroid detector with YOLO model
4. Test real-time performance

**Integration Code:**
```python
from ultralytics import YOLO

# Load trained model
model = YOLO('best.pt')

# Run inference
results = model(frame, conf=0.25, verbose=False)

# Get detection
if len(results[0].boxes) > 0:
    box = results[0].boxes[0]
    x_center = (box.xyxy[0][0] + box.xyxy[0][2]) / 2
    y_center = (box.xyxy[0][1] + box.xyxy[0][3]) / 2
    confidence = box.conf[0]
```